# Mobius export → onnx-world-model inference

Complete high-level workflow for `nvidia/Cosmos3-Edge`: text generation, image/video understanding, image/video generation, image-to-video, and action generation.

```powershell
pip install -e ~/workspace/mobius
pip install -e ~/workspace/onnx-world-model
```

`f16` keeps the package around 14 GB and runs on CUDA. Use `--dtype f32` for a
CPU-only run; that package is about 23 GB and is much slower.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import imageio.v3 as iio
import numpy as np

from onnx_world_model import WorldModel

EXPORT_DIR = Path("artifacts/cosmos3-edge-f16")
ASSET_DIR = Path("artifacts/inputs")
ASSET_DIR.mkdir(parents=True, exist_ok=True)

## 1. Export every component with one Mobius command

In [ ]:
!mobius build --model nvidia/Cosmos3-Edge {EXPORT_DIR} --features world-model --dtype f16

## 2. Load once

`providers=["cuda", "cpu"]` keeps CPU as the fallback for the handful of nodes the
CUDA provider does not implement. `graph_optimization="basic"` is required here:
ONNX Runtime's extended fusions abort while building the reasoner decoder session.

Do not export with `--ep cuda`. That pins every component to `preferred_execution_providers: ["cuda"]`,
which removes the CPU fallback and makes session creation fail.

In [ ]:
model = WorldModel.from_pretrained(
    EXPORT_DIR,
    providers=["cuda", "cpu"],
    provider_options={"cuda": {"device_id": 0}},
    graph_optimization="basic",
)
model.capabilities

Download the official Edge assets: the input frame and prompt for image-to-video,
the reference clip used by the understanding examples, and the recommended negative prompt.

In [ ]:
BASE = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets/"
IMAGE_PATH = ASSET_DIR / "edge_i2v_input.jpg"
VIDEO_PATH = ASSET_DIR / "edge_i2v_output.mp4"
PROMPT_PATH = ASSET_DIR / "example_i2v_prompt.json"
for path, name in [
    (IMAGE_PATH, "example_i2v_input.jpg"),
    (VIDEO_PATH, "diffusers_outputs/edge_i2v_diffusers.mp4"),
    (PROMPT_PATH, "example_i2v_prompt.json"),
]:
    if not path.exists():
        urlretrieve(BASE + name, path)

## 3. Text generation

In [ ]:
text = model.text.generate("Explain what a world model is in one sentence.", max_tokens=64, do_sample=False)
text.text

## 4. Image understanding

In [ ]:
image_description = model.text.generate("Describe this road scene.", image=IMAGE_PATH, max_tokens=96, do_sample=False)
image_description.text

## 5. Video understanding

In [ ]:
video_description = model.text.generate("Describe what changes in this video.", video=VIDEO_PATH, video_sample_fps=2, max_tokens=128, do_sample=False)
video_description.text

## 6. Text-to-image

In [ ]:
image = model.image.generate("A photorealistic orange cat sitting on a windowsill", height=256, width=256, guidance_scale=5.0, num_inference_steps=50, seed=42)
image.images.shape

## 7. Text-to-video

In [ ]:
text_video = model.video.generate("A robot arm moves a red block to the left", frames=5, height=256, width=256, guidance_scale=5.0, num_inference_steps=50, seed=42)
text_video.video.shape

## 8. Image-to-video

Passing `image=` activates the exported VAE encoder, conditioned latent mask, official Edge scheduler override, and classifier-free guidance.

This reproduces the official Edge example: the official input frame and prompt at
121 frames, 480x832, 24 fps. `decode_latent_chunk` decodes the latent a few frames
at a time, because one full-resolution VAE decode makes an intermediate activation
exceed what ONNX Runtime's CUDA kernels can index.

In [ ]:
import json

prompt = json.dumps(json.load(open(PROMPT_PATH)))
image_video = model.video.generate(
    prompt,
    image=IMAGE_PATH,
    negative_prompt=model.video.default_negative_prompt("image_to_video"),
    frames=121,
    height=480,
    width=832,
    fps=24,
    num_inference_steps=35,
    seed=0,
    decode_latent_chunk=6,
)
image_video.video.shape, image_video.timings

### Save it as an mp4

The decoder returns a float `NCTHW` tensor in `[-1, 1]`, so it is rescaled to
`uint8` `THWC` before encoding.

In [ ]:
def save_mp4(video, path, fps=24):
    frames = np.asarray(video, dtype=np.float32)[0]
    frames = np.clip((frames + 1.0) / 2.0, 0.0, 1.0)
    frames = np.transpose(frames, (1, 2, 3, 0))
    frames = (frames * 255.0 + 0.5).astype(np.uint8)
    iio.imwrite(path, frames, fps=fps, codec="libx264", pixelformat="yuv420p",
                output_params=["-crf", "18", "-movflags", "+faststart"])
    return path


OUTPUT_PATH = save_mp4(image_video.video, "artifacts/edge_i2v_onnx.mp4")
OUTPUT_PATH

In [ ]:
from IPython.display import Video

Video(str(OUTPUT_PATH), embed=True, width=832)

## 9. Action generation

`domain` selects the exported domain-aware action head. The returned tensor is cropped to that domain's raw action width.

In [ ]:
action = model.action.generate("Move the red block to the left.", domain="droid_lerobot", steps=16, num_inference_steps=30, seed=42)
action.actions.shape

These are all high-level modalities exposed by the Edge package: `text`, `image`, `video`, and `action`. Cosmos3-Edge does not ship the Cosmos3 Sound tokenizer, so audio generation is unavailable for this checkpoint.